# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate the incremental query behavior and SQL transformation logic that would run in Snowflake.

In [ ]:
import duckdb

con = duckdb.connect(database=':memory:')
print("DuckDB connection established.")

In [ ]:
# Simulate the Silver Iceberg Table in DuckDB
con.execute("""
    CREATE TABLE silver_condition_occurrence (
        person_id VARCHAR,
        condition_concept_id INT,
        condition_start_date DATE,
        etl_timestamp TIMESTAMP
    );
""")

# Insert mock data
con.execute("""
    INSERT INTO silver_condition_occurrence VALUES 
    ('P1', 317009, '2023-01-01', '2023-01-01 10:00:00'),
    ('P2', 201820, '2023-01-02', '2023-01-02 10:00:00'),
    ('P1', 201820, '2023-01-05', '2023-01-05 10:00:00');
""")

print("Simulated Silver Table:")
con.execute("SELECT * FROM silver_condition_occurrence").fetchdf()

In [ ]:
# Simulate Dynamic Table: 5 min Silver Aggregation
con.execute("""
    CREATE VIEW dt_condition_counts AS
    SELECT condition_concept_id, COUNT(*) as condition_count, MAX(etl_timestamp) as last_updated
    FROM silver_condition_occurrence
    GROUP BY condition_concept_id;
""")

print("Simulated Dynamic Table (dt_condition_counts):")
con.execute("SELECT * FROM dt_condition_counts").fetchdf()

In [ ]:
# Simulate Dynamic Table: 60 min Gold Aggregation
con.execute("""
    CREATE VIEW dt_patient_conditions AS
    SELECT person_id, COUNT(condition_concept_id) as total_conditions
    FROM silver_condition_occurrence
    GROUP BY person_id;
""")

print("Simulated Dynamic Table (dt_patient_conditions):")
con.execute("SELECT * FROM dt_patient_conditions").fetchdf()